[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/huggingface-nlp-certified/notebooks/day-09-question-answering.ipynb#scrollTo=a1c2e3f4)

---
# Day 9 · Question Answering — Extractive QA with SQuAD-Style Models
**certified-journeys / huggingface-nlp-certified** · Day 9 · Question Answering

> **Goal for today:** Build an extractive QA system that locates answer spans within a context passage — using the SQuAD dataset, a stride-aware preprocessing pipeline, and BERT fine-tuned for start/end position prediction.


In [ ]:
%pip install -q transformers datasets evaluate accelerate


## Step 1 · Extractive QA at a glance

In **extractive** (span extraction) QA, the model doesn't generate an answer — it identifies the start and end token positions within an existing context passage. The answer is always a verbatim substring of the context.

| Task type | Output | Example model |
|---|---|---|
| Extractive QA | (start_pos, end_pos) in context | BERT, RoBERTa, DistilBERT |
| Abstractive QA | Generated text | T5, BART, GPT |

The SQuAD (Stanford Question Answering Dataset) format is the community standard:
- `context`: passage of text
- `question`: natural language question
- `answers.text`: list of gold answer strings
- `answers.answer_start`: character-level start offset in `context`

**Reference:** [https://huggingface.co/docs/transformers/tasks/question_answering](https://huggingface.co/docs/transformers/tasks/question_answering)


In [ ]:
from datasets import load_dataset

# SQuAD v1.1 — all answers are guaranteed to appear in the context
# (SQuAD v2 adds unanswerable questions — a harder variant)
raw_datasets = load_dataset("squad")
print(raw_datasets)

# Inspect one example
sample = raw_datasets["train"][0]
print("\nContext (first 200 chars):", sample["context"][:200])
print("Question:", sample["question"])
print("Answers:", sample["answers"])


### What just happened?
- SQuAD v1.1 has ~87k training examples and ~10k validation examples.
- `answers` is a dict with two parallel lists: `text` (the answer string) and `answer_start` (character offset in `context`).
- Multiple annotators may provide different valid answers — the eval metric accepts any match.
- **Character offsets, not token offsets** — we'll convert these to token-level start/end positions during preprocessing, which is the trickiest part of the pipeline.


## Step 2 · Try the out-of-the-box QA pipeline

Before building from scratch, let's see what a production-ready QA model looks like. `deepset/roberta-base-squad2` is fine-tuned on SQuAD v2 (harder — includes unanswerable questions). The `pipeline` handles all preprocessing and postprocessing.


In [ ]:
from transformers import pipeline

# Load a production-quality extractive QA pipeline
qa_pipeline = pipeline(
    "question-answering",
    model="deepset/roberta-base-squad2",
)

context = """
The Hugging Face Transformers library was first released in 2018 under the name
pytorch-pretrained-bert. It was later renamed to Transformers and extended to
support multiple frameworks including PyTorch, TensorFlow, and JAX. The library
is maintained by Hugging Face, a company founded in 2016 and headquartered in
New York City. As of 2024, the library has over 100,000 GitHub stars.
"""

questions = [
    "When was the Hugging Face Transformers library first released?",
    "What was the original name of the Transformers library?",
    "Where is Hugging Face headquartered?",
    "How many GitHub stars does the library have?",
]

for q in questions:
    result = qa_pipeline(question=q, context=context)
    print(f"Q: {q}")
    print(f"A: {result['answer']!r}  (score: {result['score']:.3f}, "
          f"start: {result['start']}, end: {result['end']})")
    print()


### What just happened?
- The pipeline returned verbatim substrings from `context` — classic extractive QA.
- `score` is the product of the start-position softmax probability and the end-position softmax probability — a rough confidence measure.
- `start` and `end` are **character** offsets in the original context string, not token indices.
- **RoBERTa-base-squad2 handles overlapping contexts automatically** via a sliding window — the pipeline abstracts this for you; we'll implement it manually below.


## Step 3 · Preprocessing — tokenize with stride for long contexts

BERT has a 512-token limit. Long Wikipedia passages easily exceed this. The solution: a **sliding window** over the context with `stride` tokens of overlap between consecutive windows.

Why overlap? If the answer span falls near a chunk boundary, it will be fully contained in at least one of the overlapping windows. Without stride, spans at boundaries would be truncated and lost.

```
Chunk 1: [Q tokens | context tokens 0–383]
Chunk 2: [Q tokens | context tokens 256–511]  ← 128-token overlap
Chunk 3: [Q tokens | context tokens 384–end]
```


In [ ]:
from transformers import AutoTokenizer

MODEL_CHECKPOINT = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

MAX_LENGTH = 384    # max total token length (question + context + special tokens)
DOC_STRIDE = 128    # overlap between consecutive windows

def preprocess_training_examples(examples):
    """
    Tokenize question-context pairs and compute token-level start/end
    positions for the answer span.
    """
    # Strip whitespace from questions — trailing spaces cause tokenization artifacts
    questions = [q.strip() for q in examples["question"]]

    # return_overflowing_tokens=True → one example may produce multiple chunks
    # return_offsets_mapping=True → character-to-token offset map for span conversion
    tokenized = tokenizer(
        questions,
        examples["context"],
        max_length=MAX_LENGTH,
        stride=DOC_STRIDE,
        truncation="only_second",   # only truncate the context (second sequence)
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    # overflow_to_sample_mapping[i] → which original example chunk i came from
    sample_map = tokenized.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized.pop("offset_mapping")

    start_positions = []
    end_positions   = []

    for i, offsets in enumerate(offset_mapping):
        orig_idx  = sample_map[i]           # which original example
        answers   = examples["answers"][orig_idx]
        input_ids = tokenized["input_ids"][i]

        # Find the token index of [SEP] that separates question from context
        # sequence_ids: 0=question tokens, 1=context tokens, None=special tokens
        sequence_ids = tokenized.sequence_ids(i)

        # Handle examples with no answers (unanswerable — not in SQuAD v1 but
        # safe to handle)
        if len(answers["answer_start"]) == 0:
            start_positions.append(0)
            end_positions.append(0)
            continue

        # Character-level answer span from the first annotator
        char_start = answers["answer_start"][0]
        char_end   = char_start + len(answers["text"][0]) - 1

        # Find the first and last token indices that belong to the context
        ctx_start = 0
        while sequence_ids[ctx_start] != 1:
            ctx_start += 1
        ctx_end = len(input_ids) - 1
        while sequence_ids[ctx_end] != 1:
            ctx_end -= 1

        # If the answer span is completely outside this chunk, point to CLS (index 0)
        if (offsets[ctx_start][0] > char_end or
                offsets[ctx_end][1] < char_start):
            start_positions.append(0)
            end_positions.append(0)
            continue

        # Walk token offsets to find start token
        tok_start = ctx_start
        while tok_start <= ctx_end and offsets[tok_start][0] <= char_start:
            tok_start += 1
        start_positions.append(tok_start - 1)

        # Walk backward to find end token
        tok_end = ctx_end
        while tok_end >= ctx_start and offsets[tok_end][1] >= char_end:
            tok_end -= 1
        end_positions.append(tok_end + 1)

    tokenized["start_positions"] = start_positions
    tokenized["end_positions"]   = end_positions
    return tokenized

print("Preprocessing function defined.")
print(f"MAX_LENGTH={MAX_LENGTH}, DOC_STRIDE={DOC_STRIDE}")


### What just happened?
- `return_overflowing_tokens=True` with `stride=DOC_STRIDE` means one SQuAD example may produce **multiple tokenized chunks**. `sample_map` links each chunk back to its source.
- `return_offsets_mapping=True` gives us `(char_start, char_end)` for every token — this is how we convert character-level answer offsets to token-level positions.
- **`truncation='only_second'`** preserves the question in full and only cuts the context — crucial because the question sets the semantics.
- When the answer falls outside a window, we set both positions to 0 (the `[CLS]` token) — this is the standard convention for "not answerable in this chunk."


In [ ]:
# Apply preprocessing — this may expand the dataset (one example → multiple chunks)
# Use a small subset for demo speed
small_train_raw = raw_datasets["train"].select(range(1000))
small_val_raw   = raw_datasets["validation"].select(range(200))

train_dataset = small_train_raw.map(
    preprocess_training_examples,
    batched=True,
    remove_columns=small_train_raw.column_names,
)

eval_dataset = small_val_raw.map(
    preprocess_training_examples,
    batched=True,
    remove_columns=small_val_raw.column_names,
)

print(f"Training examples before preprocessing: {len(small_train_raw)}")
print(f"Training chunks after preprocessing:   {len(train_dataset)}")
print(f"Feature names: {train_dataset.column_names}")


### What just happened?
- Long contexts produce more chunks than short ones — the dataset grew from 1000 examples to (likely) 1000–1100 chunks on SQuAD's relatively short passages.
- `start_positions` and `end_positions` are now token indices pointing to the answer span within each chunk's token sequence.
- **The model will be trained on all chunks** including those where the answer is absent (position 0). This teaches it to abstain when the answer isn't present — important for robustness.
- The feature schema now includes `input_ids`, `attention_mask`, `token_type_ids`, `start_positions`, `end_positions`.


## Step 4 · Fine-tune BERT for extractive QA

`AutoModelForQuestionAnswering` adds two linear layers on top of BERT's encoder — one to predict the **start logit** and one to predict the **end logit** for each token position. During training, the loss is the sum of cross-entropy losses for start and end positions.


In [ ]:
from transformers import (
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
    DefaultDataCollator,
)

model = AutoModelForQuestionAnswering.from_pretrained(MODEL_CHECKPOINT)
print(f"Model: {MODEL_CHECKPOINT}")
print(f"Parameters: {model.num_parameters():,}")

training_args = TrainingArguments(
    output_dir="./bert-squad-finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    report_to="none",
)

# DefaultDataCollator works here because we used padding='max_length'
# All sequences are already the same length — no dynamic padding needed
data_collator = DefaultDataCollator()

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

train_result = trainer.train()
print(f"\nTraining complete! Loss: {train_result.training_loss:.4f}")


### What just happened?
- `AutoModelForQuestionAnswering` adds two weight matrices of size `(hidden_size, 1)` — one for start logits and one for end logits — on top of BERT's 768-dimensional encoder.
- **The loss is `(cross_entropy(start_logits, start_pos) + cross_entropy(end_logits, end_pos)) / 2`** — a joint objective that learns to predict both boundaries simultaneously.
- `DefaultDataCollator` is appropriate here because we pre-padded to `MAX_LENGTH=384`; all tensors are identical shape, so no dynamic padding is needed.
- The training loss is dominated by chunks where the answer is at position 0 (CLS) — the model must learn to distinguish "answer is here" from "answer is absent."


## Step 5 · Evaluate with SQuAD metrics (Exact Match and F1)

SQuAD evaluation is **answer-string level**, not token-position level:
- **Exact Match (EM):** the predicted string exactly matches any gold answer (after normalising whitespace, case, and punctuation).
- **F1:** token-level overlap between predicted and gold answer strings, averaged across examples.

This means we can't use `trainer.evaluate()` directly for SQuAD metrics — we need to convert predicted (start, end) positions back to answer strings first.


In [ ]:
import evaluate
import numpy as np
import collections

squad_metric = evaluate.load("squad")

def postprocess_qa_predictions(examples, features, raw_predictions, n_best=20):
    """
    Convert (start_logits, end_logits) pairs back to answer strings.
    For each original example, pick the best-scoring valid span across all chunks.
    """
    all_start_logits, all_end_logits = raw_predictions

    # Build a map: example_id → list of feature (chunk) indices
    # We need to re-run tokenization with offset_mapping for validation
    # (offset_mapping was removed during map(); we'll use the raw validation set)
    example_to_features = collections.defaultdict(list)
    for idx, feature in enumerate(features):
        example_to_features[feature["example_id"]].append(idx)

    predictions = {}
    for example in examples:
        ex_id      = example["id"]
        context    = example["context"]
        best_score = float("-inf")
        best_answer = ""

        for feat_idx in example_to_features[ex_id]:
            start_logits = all_start_logits[feat_idx]
            end_logits   = all_end_logits[feat_idx]
            offsets      = features[feat_idx]["offset_mapping"]
            seq_ids      = features[feat_idx]["sequence_ids"]

            # Only look at context token positions (sequence_id == 1)
            context_indices = [
                i for i, sid in enumerate(seq_ids) if sid == 1
            ]

            # Try top-n start and end positions
            top_starts = np.argsort(start_logits)[-n_best:][::-1]
            top_ends   = np.argsort(end_logits)[-n_best:][::-1]

            for start in top_starts:
                for end in top_ends:
                    if start not in context_indices:
                        continue
                    if end not in context_indices:
                        continue
                    if end < start:
                        continue
                    if end - start + 1 > 30:   # max answer length heuristic
                        continue
                    score = start_logits[start] + end_logits[end]
                    if score > best_score:
                        best_score  = score
                        char_start  = offsets[start][0]
                        char_end    = offsets[end][1]
                        best_answer = context[char_start:char_end]

        predictions[ex_id] = best_answer

    return predictions

print("postprocess_qa_predictions function defined.")


In [ ]:
# Re-tokenize validation with offset_mapping retained (for postprocessing)
def preprocess_validation_examples(examples):
    questions = [q.strip() for q in examples["question"]]
    tokenized = tokenizer(
        questions,
        examples["context"],
        max_length=MAX_LENGTH,
        stride=DOC_STRIDE,
        truncation="only_second",
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )
    sample_map = tokenized.pop("overflow_to_sample_mapping")
    # Store example_id and sequence_ids for postprocessing
    tokenized["example_id"] = []
    tokenized["sequence_ids"] = []

    for i in range(len(tokenized["input_ids"])):
        orig_idx = sample_map[i]
        tokenized["example_id"].append(examples["id"][orig_idx])
        tokenized["sequence_ids"].append(tokenized.sequence_ids(i))

    return tokenized

val_features = small_val_raw.map(
    preprocess_validation_examples,
    batched=True,
    remove_columns=small_val_raw.column_names,
)

# Run raw predictions (start_logits, end_logits) per chunk
raw_preds = trainer.predict(val_features)
print("Raw prediction shape:",
      raw_preds.predictions[0].shape,  # start_logits
      raw_preds.predictions[1].shape)  # end_logits


In [ ]:
# Convert logit predictions to answer strings
predicted_answers = postprocess_qa_predictions(
    small_val_raw,
    val_features,
    raw_preds.predictions,
)

# Format for squad metric: list of {id, prediction_text}
formatted_predictions = [
    {"id": ex_id, "prediction_text": text}
    for ex_id, text in predicted_answers.items()
]

# Format ground truth: list of {id, answers}
references = [
    {"id": ex["id"], "answers": ex["answers"]}
    for ex in small_val_raw
]

# Compute SQuAD Exact Match and F1
results = squad_metric.compute(
    predictions=formatted_predictions,
    references=references,
)
print("SQuAD Evaluation Results:")
print(f"  Exact Match: {results['exact_match']:.2f}")
print(f"  F1:          {results['f1']:.2f}")


### What just happened?
- `trainer.predict()` ran the model over all validation chunks and returned two logit arrays: one per token for start, one per token for end.
- Our postprocessing function selected the highest-scoring valid `(start, end)` pair across all chunks belonging to each original example.
- **The SQuAD metric normalises** both prediction and reference: lowercase, strip punctuation and articles, then compare. This makes `"the Amazon River"` match `"Amazon River"`.
- On the full SQuAD v1.1 dataset, BERT-base reaches ~88 F1 and ~81 EM — well above human performance on easy examples.


In [ ]:
# Save the fine-tuned QA model and do a live demo
import torch
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

QA_SAVE_DIR = "./bert-squad-final"
trainer.save_model(QA_SAVE_DIR)
print(f"Model saved to {QA_SAVE_DIR}")

# Reload and run a custom QA inference
qa_tok   = AutoTokenizer.from_pretrained(QA_SAVE_DIR)
qa_model = AutoModelForQuestionAnswering.from_pretrained(QA_SAVE_DIR)
qa_model.eval()

demo_context = """
BERT (Bidirectional Encoder Representations from Transformers) was introduced
by Google AI Language in a 2018 paper titled 'BERT: Pre-training of Deep
Bidirectional Transformers for Language Understanding'. The model was pre-trained
on Wikipedia and BooksCorpus using masked language modeling and next sentence
prediction objectives. BERT-base has 12 transformer layers and 110 million parameters.
"""
demo_question = "How many parameters does BERT-base have?"

with torch.no_grad():
    inputs  = qa_tok(demo_question, demo_context, return_tensors="pt", truncation=True)
    outputs = qa_model(**inputs)

start_idx = torch.argmax(outputs.start_logits)
end_idx   = torch.argmax(outputs.end_logits) + 1  # end is exclusive in slice
answer    = qa_tok.decode(inputs["input_ids"][0][start_idx:end_idx])

print(f"Question: {demo_question}")
print(f"Answer:   {answer!r}")


### What just happened?
- `torch.argmax(outputs.start_logits)` selects the token index with the highest start-position score.
- `tokenizer.decode(input_ids[start:end])` converts token ids back to a readable answer string — the `##` subword prefixes are merged automatically by the tokenizer.
- **This is the simplest possible postprocessing** (greedy argmax) — production systems like the `pipeline` use the full top-N search we implemented in `postprocess_qa_predictions` to handle multi-chunk contexts.
- The model reads both the question and context in a single forward pass — this is more efficient than re-encoding the context for every question (unlike bi-encoder retrieval systems).


In [ ]:
# Challenge: Handle a long context that exceeds MAX_LENGTH with stride
# Your solution here
#
# Given the long_context and question below:
#   1. Tokenize with stride=128, max_length=384, return_overflowing_tokens=True
#   2. For each chunk, run the model and collect (start_logits, end_logits)
#   3. For each chunk, find the best valid (start_tok, end_tok) in the context portion
#   4. Convert the best (start_tok, end_tok) back to character offsets using offset_mapping
#   5. Across all chunks, pick the highest-scoring answer and print it
#
# This is the manual version of what qa_pipeline does behind the scenes.

long_context = """
The Python programming language was created by Guido van Rossum and first released
in 1991. Python emphasises code readability with significant use of indentation.
It supports multiple programming paradigms including structured, object-oriented,
and functional programming.

Python is dynamically typed and garbage-collected. The language is often described
as batteries included because of its comprehensive standard library. Python's design
philosophy is captured in The Zen of Python: 'Beautiful is better than ugly,
Explicit is better than implicit, Simple is better than complex.'

The Python Software Foundation (PSF) is a non-profit organisation that holds the
intellectual property rights behind Python. The PSF was founded in 2001.
Python 3.0 was released in 2008, intentionally backwards-incompatible with Python 2.
Python consistently ranks as one of the most popular programming languages globally.
""" * 3  # Repeat to simulate a context that exceeds 384 tokens

long_question = "When was Python first released?"

# your_inputs = qa_tok(long_question, long_context, max_length=384, stride=128,
#                      truncation='only_second', return_overflowing_tokens=True,
#                      return_offsets_mapping=True, padding='max_length')
# ...


---
## Day 9 key concepts recap

| Concept | What to remember |
|---|---|
| Extractive QA | Predicts (start_token, end_token) in context — answer is always a verbatim span |
| SQuAD format | `context`, `question`, `answers.text`, `answers.answer_start` (char offset) |
| `return_overflowing_tokens=True` | Splits long contexts into overlapping chunks — each chunk is a separate model input |
| `stride` / `doc_stride` | Overlap tokens between chunks so answer spans at boundaries are always fully covered |
| `return_offsets_mapping=True` | Maps each token back to its character span — needed to convert token predictions back to text |
| `truncation='only_second'` | Truncates only the context, never the question |
| SQuAD metrics | EM = exact string match; F1 = token-level overlap; both normalise whitespace/case/punctuation |
| Postprocessing | After prediction, select best (start, end) pair across all chunks per example using logit score sum |

> **Tip:** Long contexts get truncated during tokenization — use stride (doc_stride) to create overlapping windows so answer spans near chunk boundaries are still recoverable.

---
## What's next
**Day 10** → Summarization and Seq2Seq: shift from encoder-only (BERT) to encoder-decoder architectures (BART, T5) for generative text tasks where the output is not constrained to a span of the input.

Mark Day 9 complete in your [tracker](../index.html).
